# 12 — Lógica de optimización de repostaje

**Proyecto:** RepostaPro — Optimización del repostaje en flotas comerciales  
**Autor:** Víctor González Martín  
**Notebook:** 12 — Lógica de optimización de repostaje

## Objetivo del notebook

Validar la lógica de optimización de repostaje implementada en `src/modelos.py`. Aplicamos las funciones a casos de prueba representativos antes de la aplicación masiva a las 3 flotas.

## Componentes de la lógica

1. **`distancia_haversine`**: cálculo geodésico estándar.
2. **`encontrar_estaciones_cercanas`**: filtrado por radio.
3. **`decidir_repostaje`**: comparación esperar vs repostar hoy.
4. **`optimizar_repostaje_estacionario`**: optimización para vehículos en base fija.
5. **`optimizar_repostaje_ruta`**: optimización para vehículos en movimiento.

## Casos de validación

| Caso | Tipo | Ubicación |
|---|---|---|
| 1 | Estacionario urbano | Vallecas (Madrid) |
| 2 | Estacionario interurbano | Coslada (polígono logístico) |
| 3 | Móvil largo recorrido | Ruta Madrid → Barcelona (4 waypoints) |

In [2]:
# Configuración
import sys
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.family"] = "Calibri"
plt.rcParams["font.size"] = 11
sns.set_style("whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)

# Añadir raíz del proyecto al sys.path
RAIZ_PROYECTO = Path("..").resolve()
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

# Importar funciones de optimización
from src.modelos import ( distancia_haversine,encontrar_estaciones_cercanas,
    decidir_repostaje,optimizar_repostaje_estacionario,optimizar_repostaje_ruta,MARGEN_SEGURIDAD_CTS,)

# Rutas
CARPETA_RESULTADOS = Path("../outputs/resultados_modelos")

# Cargar el dataset de predicciones de validación generado en Fase 1
df_pred = pd.read_parquet(CARPETA_RESULTADOS / "predicciones_validacion_modelo_produccion.parquet")
df_pred["fecha"] = pd.to_datetime(df_pred["fecha"])

print(f"  Dataset de predicciones cargado")
print(f"  Filas:    {len(df_pred):,}")
print(f"  Rango:    {df_pred['fecha'].min().date()} → {df_pred['fecha'].max().date()}")
print(f"  Carburantes: {df_pred['carburante'].unique().tolist()}")
print(f"  Margen seguridad configurado: {MARGEN_SEGURIDAD_CTS} cts/L")

  Dataset de predicciones cargado
  Filas:    154,976
  Rango:    2026-06-15 → 2026-06-21
  Carburantes: ['Gasóleo A', 'Gasolina 95 E5']
  Margen seguridad configurado: 1.0 cts/L


## Caso 1: Vehículo estacionario urbano (Vallecas, Madrid)

Simulamos una furgoneta de mensajería con base en Vallecas. Necesita repostar 50 litros de Gasóleo A. El sistema busca estaciones en un radio de 5 km y decide si repostar hoy (19 jun) o esperar a mañana (20 jun).

In [3]:
# CASO 1: Furgoneta mensajería urbana en Vallecas
print(">>> CASO 1: Furgoneta mensajería · Vallecas (Madrid)")
print("=" * 70)

# Parámetros del vehículo
lat_vallecas = 40.3925
lon_vallecas = -3.6650
radio_urbano = 5  # km
litros_furgoneta = 50
consumo_furgoneta = 0.10  # L/km (10 L/100km típico de furgoneta diésel)
carburante = "Gasóleo A"

# Fechas de prueba (de los 7 días de validación)
fecha_hoy = pd.Timestamp("2026-06-19")
fecha_manana = pd.Timestamp("2026-06-20")

# Optimizar
decision = optimizar_repostaje_estacionario(df_pred,lat_vallecas, lon_vallecas,radio_urbano,
    fecha_hoy, fecha_manana,litros_furgoneta, consumo_furgoneta,carburante,)

print(f"Parámetros del caso:")
print(f"  Vehículo:           Furgoneta mensajería")
print(f"  Base:               Vallecas ({lat_vallecas}, {lon_vallecas})")
print(f"  Radio búsqueda:     {radio_urbano} km")
print(f"  Litros a repostar:  {litros_furgoneta} L de {carburante}")
print(f"  Consumo vehículo:   {consumo_furgoneta} L/km (= {consumo_furgoneta*100} L/100km)")
print(f"  Fecha hoy:          {fecha_hoy.date()}")
print(f"  Fecha mañana:       {fecha_manana.date()}")
print(f"\nDecisión del sistema:")
if decision:
    print(f"  Acción:                {decision['accion'].upper()}")
    print(f"  Día:                   {decision['dia']}")
    print(f"  Estación:              ID {decision['estacion_id']} ({decision['marca']})")
    print(f"  Precio:                {decision['precio']:.3f} €/L")
    print(f"  Distancia a estación:  {decision['distancia_km']:.2f} km")
    print(f"  Coste total:           {decision['coste_total']:.2f} €")
    print(f"  Ahorro vs alternativa: {decision['ahorro_vs_alternativa']:.2f} €")
    if "ahorro_vs_naive" in decision:
        print(f"  Ahorro vs naive (peor estación radio): {decision['ahorro_vs_naive']:.2f} €")
        print(f"  Estaciones disponibles en radio:        {decision['n_estaciones_disponibles']}")
else:
    print("  No se encontraron estaciones en el radio.")

>>> CASO 1: Furgoneta mensajería · Vallecas (Madrid)
Parámetros del caso:
  Vehículo:           Furgoneta mensajería
  Base:               Vallecas (40.3925, -3.665)
  Radio búsqueda:     5 km
  Litros a repostar:  50 L de Gasóleo A
  Consumo vehículo:   0.1 L/km (= 10.0 L/100km)
  Fecha hoy:          2026-06-19
  Fecha mañana:       2026-06-20

Decisión del sistema:
  Acción:                ESPERAR
  Día:                   mañana
  Estación:              ID 14402 (SIMON GRUP)
  Precio:                1.388 €/L
  Distancia a estación:  3.13 km
  Coste total:           70.78 €
  Ahorro vs alternativa: 0.45 €
  Ahorro vs naive (peor estación radio): 10.37 €
  Estaciones disponibles en radio:        68


## Caso 2: Camión interurbano (Coslada, polígono logístico)

Simulamos un camión mediano con base en Coslada (uno de los grandes polígonos logísticos de Madrid). Necesita repostar 200 litros de Gasóleo A. El radio de búsqueda es mayor (15 km) porque los camiones tienen más flexibilidad para desviarse.

In [4]:
# CASO 2: Camión interurbano en Coslada
print(">>> CASO 2: Camión interurbano · Coslada (Madrid)")
print("=" * 70)

# Parámetros del vehículo
lat_coslada = 40.4239
lon_coslada = -3.4836
radio_interurbano = 15  # km
litros_camion = 200
consumo_camion = 0.30  # L/km (30 L/100km típico camión mediano)
carburante = "Gasóleo A"

# Fechas
fecha_hoy = pd.Timestamp("2026-06-19")
fecha_manana = pd.Timestamp("2026-06-20")

decision = optimizar_repostaje_estacionario(df_pred,lat_coslada, lon_coslada,radio_interurbano,
    fecha_hoy, fecha_manana,litros_camion, consumo_camion,carburante)

print(f"Parámetros del caso:")
print(f"  Vehículo:           Camión mediano")
print(f"  Base:               Coslada ({lat_coslada}, {lon_coslada})")
print(f"  Radio búsqueda:     {radio_interurbano} km")
print(f"  Litros a repostar:  {litros_camion} L de {carburante}")
print(f"  Consumo vehículo:   {consumo_camion} L/km (= {consumo_camion*100} L/100km)")

print(f"\nDecisión del sistema:")
if decision:
    print(f"  Acción:                {decision['accion'].upper()}")
    print(f"  Día:                   {decision['dia']}")
    print(f"  Estación:              ID {decision['estacion_id']} ({decision['marca']})")
    print(f"  Precio:                {decision['precio']:.3f} €/L")
    print(f"  Distancia a estación:  {decision['distancia_km']:.2f} km")
    print(f"  Coste total:           {decision['coste_total']:.2f} €")
    print(f"  Ahorro vs alternativa: {decision['ahorro_vs_alternativa']:.2f} €")
    if "ahorro_vs_naive" in decision:
        print(f"  Ahorro vs naive: {decision['ahorro_vs_naive']:.2f} €")
        print(f"  Estaciones en radio: {decision['n_estaciones_disponibles']}")
else:
    print("  No se encontraron estaciones en el radio.")

>>> CASO 2: Camión interurbano · Coslada (Madrid)
Parámetros del caso:
  Vehículo:           Camión mediano
  Base:               Coslada (40.4239, -3.4836)
  Radio búsqueda:     15 km
  Litros a repostar:  200 L de Gasóleo A
  Consumo vehículo:   0.3 L/km (= 30.0 L/100km)

Decisión del sistema:
  Acción:                ESPERAR
  Día:                   mañana
  Estación:              ID 13631 (PETROPRIX)
  Precio:                1.383 €/L
  Distancia a estación:  9.77 km
  Coste total:           286.65 €
  Ahorro vs alternativa: 1.35 €
  Ahorro vs naive: 67.15 €
  Estaciones en radio: 230


## Caso 3: Camión de largo recorrido (Ruta Madrid → Barcelona)

Simulamos un camión grande que sale de Madrid hacia Barcelona por la A-2 (Corredor Mediterráneo RTE-T). Necesita repostar 400 litros de Gasóleo A durante la ruta. El sistema identifica el mejor waypoint para repostar entre los 5 puntos de paso, considerando precio + coste del desvío.

**Waypoints definidos**:
- Madrid (origen): 40.4168, -3.7038
- Guadalajara: 40.6298, -3.1670
- Calatayud: 41.3494, -1.6442
- Zaragoza: 41.6488, -0.8891
- Lleida: 41.6176, 0.6200
- Barcelona (destino): 41.3851, 2.1734

In [5]:
# CASO 3: Camión largo recorrido Madrid → Barcelona CON RESTRICCIÓN DE AUTONOMÍA
print(">>> CASO 3: Camión largo recorrido · Madrid → Barcelona (Corredor Mediterráneo RTE-T)")
print("=" * 95)

waypoints_mad_bcn = [
    ("Madrid",       40.4168, -3.7038),
    ("Guadalajara",  40.6298, -3.1670),
    ("Calatayud",    41.3494, -1.6442),
    ("Zaragoza",     41.6488, -0.8891),
    ("Lleida",       41.6176,  0.6200),
    ("Barcelona",    41.3851,  2.1734),]

# Parámetros del camión
radio_ruta = 10
litros_largo = 400
consumo_largo = 0.35  # L/km (35 L/100km)
capacidad_deposito = 600  # L
nivel_inicial = 150  # L (25% del depósito al salir, práctica habitual sector)
carburante = "Gasóleo A"
fecha_ruta = pd.Timestamp("2026-06-19")

autonomia_inicial = nivel_inicial / consumo_largo

decision_ruta = optimizar_repostaje_ruta(df_pred,waypoints_mad_bcn,fecha_ruta,radio_ruta,litros_largo,consumo_largo,carburante,
    nivel_inicial_l=nivel_inicial,capacidad_deposito_l=capacidad_deposito,)

print(f"Parámetros del caso:")
print(f"  Vehículo:           Camión largo recorrido")
print(f"  Ruta:               Madrid → Barcelona (A-2, Corredor Mediterráneo RTE-T)")
print(f"  Waypoints:          {len(waypoints_mad_bcn)} puntos de paso")
print(f"  Radio por waypoint: {radio_ruta} km")
print(f"  Litros a repostar:  {litros_largo} L de {carburante}")
print(f"  Consumo vehículo:   {consumo_largo} L/km (= {consumo_largo*100} L/100km)")
print(f"  Capacidad depósito: {capacidad_deposito} L")
print(f"  Nivel inicial:      {nivel_inicial} L ({nivel_inicial/capacidad_deposito*100:.0f}% del depósito)")
print(f"  Autonomía inicial:  {autonomia_inicial:.0f} km")
print(f"\nAnálisis de viabilidad por waypoint:\n")
print(f"  {'Waypoint':<13} {'Dist. acum.':>12} {'Comb. nec.':>12} {'Viable':>8} {'Marca':>20} {'Precio':>10} {'Coste':>10}")
print("  " + "-" * 95)

for opcion in decision_ruta["todas_opciones"]:
    viable_str = "✓" if opcion.get("viable", False) and "coste_total" in opcion else "✗"
    dist = opcion.get("distancia_acumulada_km", 0)
    comb_nec = opcion.get("combustible_necesario", 0)
    
    if "coste_total" in opcion:
        marca = opcion["marca"][:20]
        precio = f"{opcion['precio']:.3f} €/L"
        coste = f"{opcion['coste_total']:.2f} €"
    else:
        marca = "—"
        precio = "—"
        coste = "—"
    
    print(f"  {opcion['waypoint']:<13} {dist:>9.0f} km {comb_nec:>9.0f} L {viable_str:>8} "
          f"{marca:>20} {precio:>10} {coste:>10}")

print(f"\n>>> DECISIÓN ÓPTIMA (considerando autonomía) <<<")
if decision_ruta["mejor_opcion"]:
    mejor = decision_ruta["mejor_opcion"]
    print(f"  Mejor waypoint:        {mejor['waypoint']}")
    print(f"  Marca:                 {mejor['marca']}")
    print(f"  Precio:                {mejor['precio']:.3f} €/L")
    print(f"  Coste total:           {mejor['coste_total']:.2f} €")
    print(f"  Ahorro vs peor viable: {decision_ruta['ahorro_vs_peor_waypoint']:.2f} €")
    print(f"  Waypoints viables:     {decision_ruta['n_viables']}")
    print(f"  Waypoints no viables:  {decision_ruta['n_no_viables']}")
else:
    print(f"  NINGÚN waypoint viable: {decision_ruta.get('error', 'desconocido')}")

>>> CASO 3: Camión largo recorrido · Madrid → Barcelona (Corredor Mediterráneo RTE-T)
Parámetros del caso:
  Vehículo:           Camión largo recorrido
  Ruta:               Madrid → Barcelona (A-2, Corredor Mediterráneo RTE-T)
  Waypoints:          6 puntos de paso
  Radio por waypoint: 10 km
  Litros a repostar:  400 L de Gasóleo A
  Consumo vehículo:   0.35 L/km (= 35.0 L/100km)
  Capacidad depósito: 600 L
  Nivel inicial:      150 L (25% del depósito)
  Autonomía inicial:  429 km

Análisis de viabilidad por waypoint:

  Waypoint       Dist. acum.   Comb. nec.   Viable                Marca     Precio      Coste
  -----------------------------------------------------------------------------------------------
  Madrid                0 km         0 L        ✓            BALLENOIL  1.399 €/L   568.45 €
  Guadalajara          60 km        21 L        ✓               BEROIL  1.299 €/L   521.03 €
  Calatayud           238 km        83 L        ✓              BONAREA  1.285 €/L   514.61 €
 